# 044 — Detección de anomalías

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Estadístico:** `z = (x−μ)/σ` con |z|>3 como referencia bajo normalidad. Los outliers
inflan σ y se esconden (*masking*): versión robusta `z = (x − mediana)/(1.4826·MAD)`.

**Isolation Forest:** las anomalías son pocas y diferentes → se aíslan con menos cortes
aleatorios. `score(x) = 2^(−E[h(x)]/c(n))`; camino corto ⇒ score → 1. Casi lineal, sin
supuesto de distribución; `contamination` fija el umbral (decisión, no propiedad).

**LOF:** compara densidad local con la de los k vecinos; LOF ≫ 1 = raro *para su región*
(anomalía local que los métodos globales no ven). Costo O(n²), sensible a k.

**Evaluación:** con 0.1-5 % de anomalías la accuracy es inútil; se usa precision/recall de
alarmas y el umbral se fija por capacidad de revisión y costos FN/FP. Outlier (error de
dato) ≠ anomalía de interés: el triage es humano.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — z-score clásico vs. robusto.** Latencias (ms):
[20, 22, 21, 19, 23, 20, 22, 21, 250]. (a) Calcula μ, σ (poblacional) y z(250).
(b) Calcula mediana, MAD y z_robusto(250). (c) ¿Con el umbral |z|>3, qué versión detecta
la anomalía y por qué?

**Ejercicio 2 — Masking.** Añade una segunda anomalía de 260 ms a la serie anterior y
recalcula z(250) clásico. ¿Sube o baja? ¿Qué implica para series con múltiples anomalías?

**Ejercicio 3 — Path length conceptual.** En un Isolation Tree sobre los montos
[10, 11, 12, 13, 500], el primer corte se elige uniforme en [10, 500]. (a) ¿Qué
probabilidad tiene el primer corte de caer en (13, 500) y aislar al 500 de inmediato?
(b) ¿Por qué el punto 12 necesitará en promedio muchos más cortes? Conecta con la fórmula
del score.

**Ejercicio 4 — Umbral operativo.** Un detector produce 10 000 scores diarios; el equipo
puede revisar 50 casos/día. Históricamente ~0.2 % de las transacciones son fraude y el
detector pone el 80 % de los fraudes en el top-1 % de scores. (a) ¿Qué umbral (percentil)
elegirías? (b) ¿Cuántos fraudes esperas en las 50 alarmas y qué precision implica?


In [ ]:
# TODO: ejecuta run_lab("ml", seed=44)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 1 y 2: z clásico vs. robusto
import statistics as st

lat = [20, 22, 21, 19, 23, 20, 22, 21, 250]

def z_clasico(x, xs):
    return None  # completa: (x − μ) / σ_poblacional

def z_robusto(x, xs):
    med = st.median(xs)
    mad = None  # completa: mediana de |xi − med|
    return (x - med) / (1.4826 * mad)

# calcula ambos para 250; luego añade 260 y recalcula el clásico


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: umbral operativo
n_diarias = 10_000
capacidad = 50
tasa_fraude = 0.002
recall_top1pct = 0.80
# ¿qué percentil de score revisar? ¿fraudes esperados en 50 alarmas? ¿precision?


## Reflexión

1. En el ejemplo trabajado, el fraude de 480 USD produce z ≈ 3.0 con estimadores clásicos
   y z ≈ 210 con mediana/MAD. ¿Qué propiedad de la media y la desviación estándar explica
   la diferencia y cómo se llama el efecto?
2. ¿Por qué `contamination=0.05` no es un hecho sobre los datos sino una decisión
   operativa, y qué información del negocio usarías para fijarla?
3. Da un ejemplo concreto de anomalía que LOF detectaría e Isolation Forest
   probablemente no, y explica por qué con el mecanismo de cada método.
